In [66]:
import pandas as pd
import zipfile
import os

# Primero, veamos qué hay dentro del ZIP de clima
zip_path = 'data/raw/clima/dataset_clima.zip'

# Extraer el contenido temporalmente para explorar
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    # Listar los archivos dentro
    file_list = zip_ref.namelist()
    print("Archivos en dataset_clima.zip:")
    for file in file_list:
        print(f"  - {file}")

# Extraer a una carpeta temporal para análisis
extract_path = '/Users/juanjo/Desktop/faq/ML/TPFINAL/data/raw/clima/temp_extracted'


print(f"\nArchivos extraídos en: {extract_path}")

Archivos en dataset_clima.zip:
  - data_stream-moda_stepType-avgua.nc
  - data_stream-moda_stepType-avgad.nc

Archivos extraídos en: /Users/juanjo/Desktop/faq/ML/TPFINAL/data/raw/clima/temp_extracted


In [67]:
import xarray as xr
import numpy as np
import pandas as pd

print("="*80)
print("EXPLORANDO LOS ARCHIVOS NETCDF DE CLIMA")
print("="*80)

# Rutas a los archivos extraídos
archivo1 = 'data/raw/clima/temp_extracted/data_stream-moda_stepType-avgua.nc'
archivo2 = 'data/raw/clima/temp_extracted/data_stream-moda_stepType-avgad.nc'

# Abrir el primer archivo
print("\n" + "="*80)
print("ARCHIVO 1: data_stream-moda_stepType-avgua.nc")
print("="*80)

ds1 = xr.open_dataset(archivo1)
print("\nInformación general del dataset:")
print(ds1)

print("\n" + "-"*80)
print("Variables disponibles:")
for var in ds1.data_vars:
    print(f"\n  Variable: {var}")
    print(f"  Dimensiones: {ds1[var].dims}")
    print(f"  Shape: {ds1[var].shape}")
    print(f"  Descripción: {ds1[var].attrs.get('long_name', 'No disponible')}")
    print(f"  Unidades: {ds1[var].attrs.get('units', 'No disponible')}")

print("\n" + "-"*80)
print("Dimensiones del archivo:")
for dim in ds1.dims:
    print(f"  {dim}: {ds1.dims[dim]} valores")

# Examinar las coordenadas temporales
if 'time' in ds1.coords:
    print("\n" + "-"*80)
    print("Información temporal:")
    times = ds1.coords['time'].values
    print(f"  Número total de pasos temporales: {len(times)}")
    print(f"  Primer registro: {pd.Timestamp(times[0])}")
    print(f"  Último registro: {pd.Timestamp(times[-1])}")
    
    # Calcular el período que cubre
    fecha_inicio = pd.Timestamp(times[0])
    fecha_fin = pd.Timestamp(times[-1])
    años_cubiertos = fecha_fin.year - fecha_inicio.year + 1
    print(f"  Período total: {años_cubiertos} años")
    
    # Verificar si es mensual
    if len(times) > 1:
        intervalo = pd.Timestamp(times[1]) - pd.Timestamp(times[0])
        print(f"  Intervalo entre registros: {intervalo}")

# Examinar las coordenadas espaciales
if 'latitude' in ds1.coords and 'longitude' in ds1.coords:
    print("\n" + "-"*80)
    print("Información espacial:")
    lats = ds1.coords['latitude'].values
    lons = ds1.coords['longitude'].values
    print(f"  Latitudes: {len(lats)} valores")
    print(f"  Rango latitudes: {lats.min():.2f}° a {lats.max():.2f}°")
    print(f"  Longitudes: {len(lons)} valores")
    print(f"  Rango longitudes: {lons.min():.2f}° a {lons.max():.2f}°")
    
    # Argentina está aproximadamente entre -55° y -22° latitud, -73° y -53° longitud
    print(f"\n  Verificación de cobertura de Argentina:")
    cubre_argentina = (lats.min() <= -22 and lats.max() >= -55 and 
                      lons.min() <= -53 and lons.max() >= -73)
    print(f"  ¿Cubre Argentina? {'Sí' if cubre_argentina else 'No'}")

# Ahora el segundo archivo
print("\n\n" + "="*80)
print("ARCHIVO 2: data_stream-moda_stepType-avgad.nc")
print("="*80)

ds2 = xr.open_dataset(archivo2)
print("\nInformación general del dataset:")
print(ds2)

print("\n" + "-"*80)
print("Variables disponibles:")
for var in ds2.data_vars:
    print(f"\n  Variable: {var}")
    print(f"  Dimensiones: {ds2[var].dims}")
    print(f"  Shape: {ds2[var].shape}")
    print(f"  Descripción: {ds2[var].attrs.get('long_name', 'No disponible')}")
    print(f"  Unidades: {ds2[var].attrs.get('units', 'No disponible')}")

# Verificar si las dimensiones temporales coinciden
print("\n" + "="*80)
print("COMPARACIÓN ENTRE ARCHIVOS")
print("="*80)

print("\nDimensiones temporales:")
print(f"  Archivo 1: {ds1.dims.get('time', 'No tiene dimensión time')} registros")
print(f"  Archivo 2: {ds2.dims.get('time', 'No tiene dimensión time')} registros")

if 'time' in ds1.coords and 'time' in ds2.coords:
    coinciden = np.array_equal(ds1.coords['time'].values, ds2.coords['time'].values)
    print(f"  ¿Las fechas coinciden? {'Sí' if coinciden else 'No'}")

print("\nDimensiones espaciales:")
if 'latitude' in ds1.coords and 'latitude' in ds2.coords:
    print(f"  Archivo 1 - Latitudes: {len(ds1.coords['latitude'])} | Longitudes: {len(ds1.coords['longitude'])}")
    print(f"  Archivo 2 - Latitudes: {len(ds2.coords['latitude'])} | Longitudes: {len(ds2.coords['longitude'])}")

EXPLORANDO LOS ARCHIVOS NETCDF DE CLIMA

ARCHIVO 1: data_stream-moda_stepType-avgua.nc

Información general del dataset:
<xarray.Dataset> Size: 32MB
Dimensions:     (valid_time: 682, latitude: 137, longitude: 85)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 5kB 1969-01-01 ... 2025-10-01
  * latitude    (latitude) float64 1kB -21.0 -21.25 -21.5 ... -54.5 -54.75 -55.0
  * longitude   (longitude) float64 680B -74.0 -73.75 -73.5 ... -53.25 -53.0
    number      int64 8B ...
    expver      (valid_time) <U4 11kB ...
Data variables:
    t2m         (valid_time, latitude, longitude) float32 32MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-12-05T23:38 GRIB to CDM+CF via cfgrib-0.9.1...

---------------------------------

/var/folders/k8/kzd29bps4d3dmdpdqk668cph0000gn/T/ipykernel_39136/1349247408.py:34: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  {dim}: {ds1.dims[dim]} valores")
/var/folders/k8/kzd29bps4d3dmdpdqk668cph0000gn/T/ipykernel_39136/1349247408.py:97: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  Archivo 1: {ds1.dims.get('time', 'No tiene dimensión time')} registros")
<frozen _collections_abc>:811: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapp

In [68]:
import geopandas as gpd
import pandas as pd
import numpy as np

print("="*80)
print("EXTRACCIÓN DE COORDENADAS DE DEPARTAMENTOS - VERSIÓN CORREGIDA")
print("="*80)

# Cargar el shapefile de departamentos
shapefile_path = 'data/raw/geograficos/departamentos.shp'
gdf_deptos = gpd.read_file(shapefile_path)

print(f"\n📊 Información del shapefile:")
print(f"   Total de registros: {len(gdf_deptos)}")
print(f"   Sistema de coordenadas: {gdf_deptos.crs}")

# Usar directamente las coordenadas precalculadas y los nombres de departamento
df_coordenadas = gdf_deptos[['prov_nombre', 'nombre', 'centr_lat', 'centr_lon']].copy()

# Renombrar columnas para consistencia
df_coordenadas.columns = ['provincia_nombre', 'departamento_nombre', 'lat', 'lon']

print(f"\n🔍 Verificando tipos de datos originales...")
print(f"   Tipo de 'lat': {df_coordenadas['lat'].dtype}")
print(f"   Tipo de 'lon': {df_coordenadas['lon'].dtype}")
print(f"\n   Ejemplos de valores raw:")
print(f"   Lat: {df_coordenadas['lat'].iloc[0]}")
print(f"   Lon: {df_coordenadas['lon'].iloc[0]}")

# Convertir a numérico, forzando errores a NaN
print(f"\n🔄 Convirtiendo coordenadas a tipo numérico...")
df_coordenadas['lat'] = pd.to_numeric(df_coordenadas['lat'], errors='coerce')
df_coordenadas['lon'] = pd.to_numeric(df_coordenadas['lon'], errors='coerce')

print(f"   Tipo de 'lat' después de conversión: {df_coordenadas['lat'].dtype}")
print(f"   Tipo de 'lon' después de conversión: {df_coordenadas['lon'].dtype}")

# Verificar y limpiar datos
print(f"\n🔍 Verificando calidad de datos...")
print(f"   Registros totales: {len(df_coordenadas)}")
print(f"   Departamentos únicos: {df_coordenadas['departamento_nombre'].nunique()}")
print(f"   Provincias únicas: {df_coordenadas['provincia_nombre'].nunique()}")

# Verificar valores faltantes
missing_coords = df_coordenadas[['lat', 'lon']].isna().sum()
if missing_coords.any():
    print(f"\n⚠️  Valores faltantes encontrados:")
    print(f"   Latitudes faltantes: {missing_coords['lat']}")
    print(f"   Longitudes faltantes: {missing_coords['lon']}")
    
    # Mostrar cuáles son los departamentos con problemas
    problematicos = df_coordenadas[df_coordenadas[['lat', 'lon']].isna().any(axis=1)]
    if len(problematicos) > 0:
        print(f"\n   Departamentos con coordenadas faltantes:")
        for _, row in problematicos[['provincia_nombre', 'departamento_nombre']].iterrows():
            print(f"      {row['provincia_nombre']} - {row['departamento_nombre']}")
    
    # Eliminar filas con coordenadas faltantes
    df_coordenadas = df_coordenadas.dropna(subset=['lat', 'lon'])
    print(f"\n   Registros después de limpiar: {len(df_coordenadas)}")

# Verificar duplicados de departamentos
duplicados = df_coordenadas.duplicated(subset=['departamento_nombre'], keep=False)
if duplicados.any():
    print(f"\n⚠️  Departamentos con nombres duplicados: {duplicados.sum()}")
    print(f"\nPrimeros 20 departamentos duplicados:")
    dups = df_coordenadas[duplicados].sort_values('departamento_nombre')
    print(dups[['provincia_nombre', 'departamento_nombre']].head(20).to_string(index=False))
    
    if duplicados.sum() > 20:
        print(f"   ... y {duplicados.sum() - 20} más")
    
    # Para departamentos con mismo nombre en diferentes provincias,
    # vamos a crear un identificador único combinando provincia + departamento
    print(f"\n   Creando identificador único provincia-departamento...")
    df_coordenadas['departamento_id'] = (
        df_coordenadas['provincia_nombre'] + ' - ' + df_coordenadas['departamento_nombre']
    )
else:
    df_coordenadas['departamento_id'] = df_coordenadas['departamento_nombre']

print(f"\n📍 Estadísticas de coordenadas:")
print(f"   Latitudes:")
print(f"      Mínimo: {df_coordenadas['lat'].min():.4f}° (punto más sur)")
print(f"      Máximo: {df_coordenadas['lat'].max():.4f}° (punto más norte)")
print(f"      Media:  {df_coordenadas['lat'].mean():.4f}°")
print(f"\n   Longitudes:")
print(f"      Mínimo: {df_coordenadas['lon'].min():.4f}° (punto más este)")
print(f"      Máximo: {df_coordenadas['lon'].max():.4f}° (punto más oeste)")
print(f"      Media:  {df_coordenadas['lon'].mean():.4f}°")

# Verificar que las coordenadas están en el rango esperado para Argentina
lat_ok = (df_coordenadas['lat'].min() >= -56) and (df_coordenadas['lat'].max() <= -21)
lon_ok = (df_coordenadas['lon'].min() >= -74) and (df_coordenadas['lon'].max() <= -53)

if lat_ok and lon_ok:
    print(f"\n   ✅ Las coordenadas están dentro del rango geográfico de Argentina")
else:
    print(f"\n   ⚠️  Advertencia: Algunas coordenadas fuera del rango esperado")

print(f"\n📋 Ejemplos de coordenadas por provincia:")
# Mostrar 2 departamentos de algunas provincias
for prov in sorted(df_coordenadas['provincia_nombre'].unique())[:5]:
    print(f"\n   {prov}:")
    ejemplos = df_coordenadas[df_coordenadas['provincia_nombre'] == prov].head(2)
    for _, row in ejemplos.iterrows():
        print(f"      {row['departamento_nombre']:30s} | Lat: {row['lat']:8.4f} | Lon: {row['lon']:8.4f}")

# Guardar coordenadas
output_path = 'data/raw/intermediate/departamentos_coordenadas.csv'
df_coordenadas.to_csv(output_path, index=False)
print(f"\n💾 Coordenadas guardadas en: {output_path}")

print(f"\n📊 Resumen del archivo guardado:")
print(f"   Registros totales: {len(df_coordenadas)}")
print(f"   Columnas: {list(df_coordenadas.columns)}")

# Mostrar un resumen de departamentos por provincia
print(f"\n📊 Departamentos por provincia:")
deptos_por_prov = df_coordenadas.groupby('provincia_nombre').size().sort_values(ascending=False)
for prov, count in deptos_por_prov.head(10).items():
    print(f"   {prov:25s}: {count:3d} departamentos")
if len(deptos_por_prov) > 10:
    print(f"   ... y {len(deptos_por_prov) - 10} provincias más")

print(f"\n{'='*80}")
print(f"EXTRACCIÓN DE COORDENADAS COMPLETADA EXITOSAMENTE")
print(f"{'='*80}")

EXTRACCIÓN DE COORDENADAS DE DEPARTAMENTOS - VERSIÓN CORREGIDA

📊 Información del shapefile:
   Total de registros: 529
   Sistema de coordenadas: EPSG:4326

🔍 Verificando tipos de datos originales...
   Tipo de 'lat': object
   Tipo de 'lon': object

   Ejemplos de valores raw:
   Lat: -37.9646159068483
   Lon: -60.2482821323384

🔄 Convirtiendo coordenadas a tipo numérico...
   Tipo de 'lat' después de conversión: float64
   Tipo de 'lon' después de conversión: float64

🔍 Verificando calidad de datos...
   Registros totales: 529
   Departamentos únicos: 447
   Provincias únicas: 24

⚠️  Departamentos con nombres duplicados: 129

Primeros 20 departamentos duplicados:
   provincia_nombre departamento_nombre
         RÃ­o Negro          25 de Mayo
           San Juan          25 de Mayo
           Misiones          25 de Mayo
       Buenos Aires          25 de Mayo
              Chaco          25 de Mayo
           San Juan          9 de Julio
         RÃ­o Negro          9 de Julio
    

In [69]:
import xarray as xr
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
from datetime import datetime

print("="*80)
print("EXTRACCIÓN DE DATOS CLIMÁTICOS MENSUALES - VERSIÓN CORREGIDA")
print("="*80)
print(f"\nHora de inicio: {datetime.now().strftime('%H:%M:%S')}\n")

# ════════════════════════════════════════════════════════════════
# PASO 1: Cargar datos
# ════════════════════════════════════════════════════════════════
print("[1/6] Cargando datasets...")

df_deptos = pd.read_csv('data/raw/intermediate/departamentos_coordenadas.csv')
print(f"   ✓ Departamentos cargados: {len(df_deptos)}")

ds_temp = xr.open_dataset('data/raw/clima/temp_extracted/data_stream-moda_stepType-avgua.nc')
ds_prec = xr.open_dataset('data/raw/clima/temp_extracted/data_stream-moda_stepType-avgad.nc')
print(f"   ✓ Datos climáticos cargados")

# ════════════════════════════════════════════════════════════════
# PASO 2: Construir grilla ERA5
# ════════════════════════════════════════════════════════════════
print("\n[2/6] Construyendo grilla ERA5...")

era5_lats = ds_temp.latitude.values
era5_lons = ds_temp.longitude.values

era5_coords = []
for lat in era5_lats:
    for lon in era5_lons:
        era5_coords.append([lat, lon])
era5_coords = np.array(era5_coords)

print(f"   ✓ Grilla ERA5: {len(era5_coords):,} puntos")

# ════════════════════════════════════════════════════════════════
# PASO 3: Emparejar departamentos con puntos ERA5
# ════════════════════════════════════════════════════════════════
print("\n[3/6] Emparejando departamentos con grilla ERA5...")

tree = cKDTree(era5_coords)
depto_coords = df_deptos[['lat', 'lon']].values
distances, indices = tree.query(depto_coords)
distances_km = distances * 111

df_deptos['era5_idx'] = indices
df_deptos['era5_lat'] = era5_coords[indices, 0]
df_deptos['era5_lon'] = era5_coords[indices, 1]

print(f"   ✓ Emparejamiento completado")
print(f"     Distancia promedio: {distances_km.mean():.2f} km")

# ════════════════════════════════════════════════════════════════
# PASO 4: Extraer datos - MÉTODO CORREGIDO
# ════════════════════════════════════════════════════════════════
print("\n[4/6] Extrayendo series temporales (método corregido)...")

# Primero, probemos con UN SOLO departamento para debuggear
print("\n   DEBUG: Probando extracción con un departamento...")

depto_test = df_deptos.iloc[0]
lat_test = depto_test['era5_lat']
lon_test = depto_test['era5_lon']

print(f"   Departamento de prueba: {depto_test['departamento_nombre']}")
print(f"   Coordenadas ERA5: lat={lat_test:.4f}, lon={lon_test:.4f}")

try:
    # Extraer temperatura
    temp_data = ds_temp.sel(latitude=lat_test, longitude=lon_test, method='nearest')
    print(f"   ✓ Temperatura extraída")
    print(f"     Variable: {list(temp_data.data_vars)}")
    print(f"     Dimensiones: {list(temp_data.dims)}")
    
    # Extraer precipitación  
    prec_data = ds_prec.sel(latitude=lat_test, longitude=lon_test, method='nearest')
    print(f"   ✓ Precipitación extraída")
    
    # Convertir a DataFrame
    temp_df = temp_data['t2m'].to_dataframe().reset_index()
    prec_df = prec_data['tp'].to_dataframe().reset_index()
    
    print(f"\n   Estructura de DataFrames extraídos:")
    print(f"   Temperatura - Columnas: {list(temp_df.columns)}")
    print(f"   Temperatura - Shape: {temp_df.shape}")
    print(f"   Precipitación - Columnas: {list(prec_df.columns)}")
    print(f"   Precipitación - Shape: {prec_df.shape}")
    
    print(f"\n   Primeras filas de temperatura:")
    print(temp_df.head(3))
    
    print(f"\n   Primeras filas de precipitación:")
    print(prec_df.head(3))
    
except Exception as e:
    print(f"   ✗ Error en extracción de prueba: {e}")
    import traceback
    traceback.print_exc()

# Si la prueba funcionó, continuar con todos los departamentos
print(f"\n   Procediendo con TODOS los departamentos...")

registros_clima = []
errores = 0

for idx, row in df_deptos.iterrows():
    try:
        # Extraer datos para este departamento
        lat_era5 = row['era5_lat']
        lon_era5 = row['era5_lon']
        
        # Seleccionar punto espacial
        temp_point = ds_temp.sel(latitude=lat_era5, longitude=lon_era5, method='nearest')
        prec_point = ds_prec.sel(latitude=lat_era5, longitude=lon_era5, method='nearest')
        
        # Convertir a DataFrames
        temp_df = temp_point['t2m'].to_dataframe().reset_index()
        prec_df = prec_point['tp'].to_dataframe().reset_index()
        
        # Crear DataFrame combinado
        clima_df = pd.DataFrame({
            'fecha': temp_df['valid_time'],
            'temperatura_kelvin': temp_df['t2m'].values,
            'precipitacion_metros': prec_df['tp'].values
        })
        
        # Agregar identificadores
        # Agregar identificadores (normalizar a MAYÚSCULAS)
        clima_df['provincia_nombre'] = row['provincia_nombre']
        clima_df['departamento_nombre'] = row['departamento_nombre']
        clima_df['departamento_id'] = str(row['departamento_id']).upper()
        
        registros_clima.append(clima_df)
        
    except Exception as e:
        errores += 1
        if errores <= 5:  # Mostrar solo los primeros 5 errores
            print(f"   ✗ Error en {row['departamento_nombre']}: {e}")
    
    if (idx + 1) % 50 == 0:
        print(f"      Procesados {idx + 1}/{len(df_deptos)} (errores: {errores})")

print(f"\n   ✓ Extracción completada")
print(f"     Departamentos exitosos: {len(registros_clima)}")
print(f"     Errores: {errores}")

if len(registros_clima) == 0:
    print(f"\n   ✗✗✗ ERROR CRÍTICO: No se pudo extraer ningún dato")
    print(f"   Deteniendo ejecución para revisar el problema")
else:
    print(f"\n   ✓ Continuando con consolidación...")

# ════════════════════════════════════════════════════════════════
# PASO 5: Consolidar
# ════════════════════════════════════════════════════════════════
if len(registros_clima) > 0:
    print("\n[5/6] Consolidando datos...")
    
    df_clima_mensual = pd.concat(registros_clima, ignore_index=True)
    
    # Convertir unidades
    df_clima_mensual['temperatura_celsius'] = df_clima_mensual['temperatura_kelvin'] - 273.15
    df_clima_mensual['precipitacion_mm'] = df_clima_mensual['precipitacion_metros'] * 1000
    
    # Extraer año y mes
    df_clima_mensual['año'] = df_clima_mensual['fecha'].dt.year
    df_clima_mensual['mes'] = df_clima_mensual['fecha'].dt.month
    
    # Seleccionar columnas finales
    df_clima_mensual = df_clima_mensual[[
        'provincia_nombre', 'departamento_nombre', 'departamento_id',
        'año', 'mes', 'fecha',
        'temperatura_celsius', 'precipitacion_mm'
    ]]
    
    print(f"   ✓ Registros finales: {len(df_clima_mensual):,}")
    
    # ════════════════════════════════════════════════════════════════
    # PASO 6: Guardar
    # ════════════════════════════════════════════════════════════════
    print("\n[6/6] Guardando...")
    
    print(f"\n📊 Estadísticas finales:")
    print(f"   Registros: {len(df_clima_mensual):,}")
    print(f"   Período: {df_clima_mensual['año'].min()} - {df_clima_mensual['año'].max()}")
    print(f"   Departamentos: {df_clima_mensual['departamento_id'].nunique()}")
    
    print(f"\n📋 Muestra de datos:")
    print(df_clima_mensual.head(10))
    
    # Guardar
    output_path = 'data/raw/processed/clima_mensual_por_departamento.csv'
    df_clima_mensual.to_csv(output_path, index=False)
    print(f"\n💾 Guardado en: {output_path}")


    print(f"\n💾 Dataset guardado exitosamente en:")
    print(f"   {output_path}")
    print(f"   Tamaño del archivo: {df_clima_mensual.memory_usage(deep=True).sum() / 1024**2:.2f} MB en memoria")

    
    print(f"\n{'='*80}")
    print("COMPLETADO EXITOSAMENTE")
    print(f"{'='*80}")

else:
    print("\n✗✗✗ NO SE GENERÓ NINGÚN DATO - REVISAR ERRORES ARRIBA")

EXTRACCIÓN DE DATOS CLIMÁTICOS MENSUALES - VERSIÓN CORREGIDA

Hora de inicio: 23:58:48

[1/6] Cargando datasets...
   ✓ Departamentos cargados: 529
   ✓ Datos climáticos cargados

[2/6] Construyendo grilla ERA5...
   ✓ Grilla ERA5: 11,645 puntos

[3/6] Emparejando departamentos con grilla ERA5...
   ✓ Emparejamiento completado
     Distancia promedio: 16.55 km

[4/6] Extrayendo series temporales (método corregido)...

   DEBUG: Probando extracción con un departamento...
   Departamento de prueba: Adolfo Gonzales Chaves
   Coordenadas ERA5: lat=-38.0000, lon=-60.2500
   ✓ Temperatura extraída
     Variable: ['t2m']
     Dimensiones: ['valid_time']
   ✓ Precipitación extraída

   Estructura de DataFrames extraídos:
   Temperatura - Columnas: ['valid_time', 'number', 'latitude', 'longitude', 'expver', 't2m']
   Temperatura - Shape: (682, 6)
   Precipitación - Columnas: ['valid_time', 'number', 'latitude', 'longitude', 'expver', 'tp']
   Precipitación - Shape: (682, 6)

   Primeras filas d

In [70]:
print("="*80)
print("ANÁLISIS EXHAUSTIVO DEL DATASET DE CULTIVOS")
print("="*80)

# Intentar cargar con diferentes codificaciones
codificaciones = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252']
df_cultivos = None

for encoding in codificaciones:
    try:
        print(f"\nIntentando cargar con codificación: {encoding}")
        df_cultivos = pd.read_csv('data/raw/cultivos/estimaciones-agricolas-2023-10.csv', 
                                  sep=';',  # ← ✅ ÚNICA LÍNEA AGREGADA
                                  encoding=encoding, 
                                  low_memory=False)
        print(f"   ✓ Éxito con codificación: {encoding}")
        break
    except UnicodeDecodeError:
        print(f"   ✗ Falló con {encoding}")
        continue

if df_cultivos is None:
    print("\n✗ No se pudo cargar el archivo")
else:
    # ✅ Limpiar nombres de columnas (quitar comillas si las hay)
    df_cultivos.columns = df_cultivos.columns.str.strip().str.replace('"', '')
    
    print(f"\n📊 INFORMACIÓN GENERAL:")
    print(f"   Filas totales: {len(df_cultivos):,}")
    print(f"   Columnas totales: {len(df_cultivos.columns)}")
    
    print(f"\n📋 LISTA DE COLUMNAS:")
    for i, col in enumerate(df_cultivos.columns, 1):
        print(f"   {i:2d}. {col}")
    
    print(f"\n📊 ANÁLISIS DETALLADO DE COLUMNAS CLAVE:")
    
    # Buscar y analizar columna de año/campaña
    col_año = None
    for col in df_cultivos.columns:
        if 'año' in col.lower() or 'anio' in col.lower() or 'campana' in col.lower() or 'campaña' in col.lower():
            col_año = col
            break
    
    if col_año:
        print(f"\n📅 COLUMNA DE CAMPAÑA: '{col_año}'")
        print(f"   Tipo de dato: {df_cultivos[col_año].dtype}")
        
        # ✅ Parsear campañas (ej: "1969/70" → años)
        def extraer_año(val):
            try:
                if '/' in str(val):
                    año_str = str(val).split('/')[1]
                    if len(año_str) == 2:
                        año_int = int(año_str)
                        return 2000 + año_int if año_int < 50 else 1900 + año_int
                    return int(año_str)
                return int(val)
            except:
                return None
        
        años = df_cultivos[col_año].apply(extraer_año).dropna()
        if len(años) > 0:
            print(f"   Rango: {int(años.min())} - {int(años.max())}")
            print(f"   Años únicos: {años.nunique()}")
    
    # Buscar y analizar columna de cultivo
    col_cultivo = None
    for col in df_cultivos.columns:
        if 'cultivo' in col.lower():
            col_cultivo = col
            break
    
    if col_cultivo:
        print(f"\n🌾 COLUMNA DE CULTIVO: '{col_cultivo}'")
        print(f"   Tipo de dato: {df_cultivos[col_cultivo].dtype}")
        cultivos = df_cultivos[col_cultivo].value_counts()
        print(f"   Cultivos únicos: {len(cultivos)}")
        print(f"\n   Top 15 cultivos:")
        for cultivo, count in cultivos.head(15).items():
            cultivo_str = str(cultivo) if pd.notna(cultivo) else "N/A"
            print(f"      {cultivo_str:40s}: {count:,} registros")
    
    # Buscar columnas geográficas
    col_provincia = None
    col_depto = None
    
    for col in df_cultivos.columns:
        if 'provincia' in col.lower() and col_provincia is None:
            col_provincia = col
        if 'departamento' in col.lower() and col_depto is None:
            col_depto = col
    
    if col_provincia:
        print(f"\n🗺️  COLUMNA DE PROVINCIA: '{col_provincia}'")
        print(f"   Tipo: {df_cultivos[col_provincia].dtype}")
        print(f"   Provincias únicas: {df_cultivos[col_provincia].nunique()}")
    
    if col_depto:
        print(f"\n🗺️  COLUMNA DE DEPARTAMENTO: '{col_depto}'")
        print(f"   Tipo: {df_cultivos[col_depto].dtype}")
        print(f"   Departamentos únicos: {df_cultivos[col_depto].nunique()}")
    
    # Buscar variables de producción
    print(f"\n📈 VARIABLES DE PRODUCCIÓN:")
    vars_prod = []
    for col in df_cultivos.columns:
        if any(x in col.lower() for x in ['rendimiento', 'rinde', 'produccion', 'superficie', 'cosecha', 'sembr']):
            vars_prod.append(col)
    
    for var in vars_prod:
        print(f"\n   📌 {var}")
        print(f"      Tipo: {df_cultivos[var].dtype}")
        valores = df_cultivos[var].dropna()
        if len(valores) > 0:
            try:
                valores_num = pd.to_numeric(valores, errors='coerce').dropna()
                if len(valores_num) > 0:
                    print(f"      No nulos: {len(valores_num):,} ({len(valores_num)/len(df_cultivos)*100:.1f}%)")
                    print(f"      Rango: {valores_num.min():,.1f} - {valores_num.max():,.1f}")
                    print(f"      Media: {valores_num.mean():,.1f}")
            except:
                print(f"      No se pudo convertir a numérico")
    
    print(f"\n\n{'='*80}")
    print("MUESTRA DE LAS PRIMERAS 10 FILAS")
    print(f"{'='*80}")
    print(df_cultivos.head(10))
    
    print(f"\n\n{'='*80}")
    print("ANÁLISIS COMPLETADO")
    print(f"{'='*80}")

ANÁLISIS EXHAUSTIVO DEL DATASET DE CULTIVOS

Intentando cargar con codificación: utf-8
   ✗ Falló con utf-8

Intentando cargar con codificación: latin-1
   ✓ Éxito con codificación: latin-1

📊 INFORMACIÓN GENERAL:
   Filas totales: 160,536
   Columnas totales: 12

📋 LISTA DE COLUMNAS:
    1. ID Provincia
    2. Provincia
    3. ID Departamento
    4. Departamento
    5. Id Cultivo
    6. Cultivo
    7. ID Campaña
    8. Campana
    9. Sup. Sembrada (Ha)
   10. Sup. Cosechada (Ha)
   11. Producción (Tn)
   12. Rendimiento (Kg/Ha)

📊 ANÁLISIS DETALLADO DE COLUMNAS CLAVE:

📅 COLUMNA DE CAMPAÑA: 'ID Campaña'
   Tipo de dato: int64
   Rango: 1 - 58
   Años únicos: 56

🌾 COLUMNA DE CULTIVO: 'Id Cultivo'
   Tipo de dato: int64
   Cultivos únicos: 41

   Top 15 cultivos:
      32                                      : 17,725 registros
      28                                      : 13,292 registros
      26                                      : 12,135 registros
      25                       

In [71]:
import pandas as pd
import numpy as np

print("="*80)
print("PASO 1: LIMPIEZA Y ESTANDARIZACIÓN DE CULTIVOS")
print("="*80)

# 1. Recargar el dataset (con separador correcto)
df_cultivos = pd.read_csv('data/raw/cultivos/estimaciones-agricolas-2023-10.csv', 
                          sep=';',  # ← ✅ AGREGADO
                          encoding='latin-1', 
                          low_memory=False)

# ✅ Limpiar nombres de columnas (quitar comillas)
df_cultivos.columns = df_cultivos.columns.str.strip().str.replace('"', '')

# 2. Generar el ID único de departamento (Crucial para el futuro Merge)
# Usamos el formato "Provincia - Departamento" para evitar duplicados
# Normalizar a MAYÚSCULAS para hacer match con clima
df_cultivos['departamento_id'] = (df_cultivos['Provincia'] + ' - ' + df_cultivos['Departamento']).str.upper()

# 3. Limpieza de AÑO (Parsear columna 'ciclo')
# Convertimos "1969/1970" -> 1970 (Tomamos el año de cosecha/cierre)
def extraer_anio(valor):
    try:
        if isinstance(valor, str) and '/' in valor:
            año_str = valor.split('/')[1]
            # Manejar años de 2 dígitos
            if len(año_str) == 2:
                año_int = int(año_str)
                return 2000 + año_int if año_int < 50 else 1900 + año_int
            return int(año_str)
        return np.nan
    except:
        return np.nan

df_cultivos['año'] = df_cultivos['Campana'].apply(extraer_anio)

# 4. Limpieza de variables numéricas (Rendimiento tenía tipo 'object')
cols_numericas = ['Rendimiento (Kg/Ha)', 'Producción (Tn)', 'Sup. Sembrada (Ha)', 'Sup. Cosechada (Ha)']

for col in cols_numericas:
    # 'coerce' convierte errores (textos no numéricos) en NaN
    df_cultivos[col] = pd.to_numeric(df_cultivos[col], errors='coerce')

# 5. Filtrado final de columnas y filas
# Nos quedamos solo con lo que definiste en el plan
df_cultivos_clean = pd.DataFrame({
    'provincia_nombre': df_cultivos['Provincia'],
    'departamento_nombre': df_cultivos['Departamento'],
    'departamento_id': df_cultivos['departamento_id'],
    'cultivo': df_cultivos['Cultivo'],
    'año': df_cultivos['año'],
    'rendimiento': df_cultivos['Rendimiento (Kg/Ha)'],
    'produccion': df_cultivos['Producción (Tn)'],
    'sup_sembrada': df_cultivos['Sup. Sembrada (Ha)']
})

# Eliminar registros que no tengan año o cultivo
df_cultivos_clean = df_cultivos_clean.dropna(subset=['año', 'cultivo'])
df_cultivos_clean['año'] = df_cultivos_clean['año'].astype(int)

# --- VERIFICACIÓN DEL RESULTADO ---
print(f"\n✅ Dataset limpio generado: df_cultivos_clean")
print(f"   Dimensiones: {df_cultivos_clean.shape}")
print(f"   Rango temporal: {df_cultivos_clean['año'].min()} - {df_cultivos_clean['año'].max()}")

print("\n📋 Muestra de datos procesados:")
print(df_cultivos_clean[['departamento_id', 'cultivo', 'año', 'rendimiento']].head())

print("\n📊 Top 10 Cultivos más frecuentes:")
print(df_cultivos_clean['cultivo'].value_counts().head(10))

print("\n⚠️ Chequeo de nulos en Rendimiento:")
nulos_rend = df_cultivos_clean['rendimiento'].isna().sum()
print(f"   Registros sin datos de rendimiento: {nulos_rend} ({nulos_rend/len(df_cultivos_clean):.1%})")

PASO 1: LIMPIEZA Y ESTANDARIZACIÓN DE CULTIVOS

✅ Dataset limpio generado: df_cultivos_clean
   Dimensiones: (160536, 8)
   Rango temporal: 1970 - 2025

📋 Muestra de datos procesados:
                         departamento_id cultivo   año  rendimiento
0              BUENOS AIRES - 25 DE MAYO     Ajo  1970         3333
1              BUENOS AIRES - 25 DE MAYO     Ajo  1971         3000
2  BUENOS AIRES - ADOLFO GONZALES CHAVES     Ajo  1970         5467
3  BUENOS AIRES - ADOLFO GONZALES CHAVES     Ajo  1971         5500
4  BUENOS AIRES - ADOLFO GONZALES CHAVES     Ajo  1972         5500

📊 Top 10 Cultivos más frecuentes:
cultivo
Maíz                17725
Trigo total         13292
Sorgo               12135
Soja total          11983
Avena               11534
Girasol             10647
Cebada forrajera     6572
Soja 1ra             6474
Centeno              6045
Papa total           5628
Name: count, dtype: int64

⚠️ Chequeo de nulos en Rendimiento:
   Registros sin datos de rendimiento: 0 (

In [72]:
import pandas as pd
import numpy as np

# Verificar que df_cultivos_clean exista, si no, se debe volver a cargar/generar
# Asumiendo que la variable de la celda anterior persiste:

print("="*80)
print("PASO 2: EXPANSIÓN DE CULTIVOS DE ANUAL A MENSUAL")
print("="*80)

# Crear un DataFrame auxiliar con los 12 meses (1 a 12)
df_meses = pd.DataFrame({'mes': range(1, 13)})
df_meses['key'] = 1

# Crear la clave de join en el DataFrame de cultivos
df_cultivos_clean['key'] = 1

# Realizar el cross-merge para duplicar cada fila 12 veces
# El rendimiento anual se 'broadcasts' (repite) para cada mes del año de cosecha.
df_cultivos_mensual = pd.merge(df_cultivos_clean, df_meses, on='key').drop('key', axis=1)

# --- VERIFICACIÓN DEL RESULTADO ---
print("\n✅ Expansión de Cultivos de ANUAL a MENSUAL completada")
print(f"   Dimensiones finales: {df_cultivos_mensual.shape}")

print("\n📋 Muestra de la Expansión (Verificar la columna 'mes'):")

# Seleccionamos un departamento y mostramos 13 filas para ver el ciclo 1-12 y el inicio del siguiente
depto_muestra = df_cultivos_mensual['departamento_id'].iloc[0]
print(f"   Departamento de muestra: {depto_muestra}")
print(df_cultivos_mensual[df_cultivos_mensual['departamento_id'] == depto_muestra].head(13).to_string(index=False))

PASO 2: EXPANSIÓN DE CULTIVOS DE ANUAL A MENSUAL

✅ Expansión de Cultivos de ANUAL a MENSUAL completada
   Dimensiones finales: (1926432, 9)

📋 Muestra de la Expansión (Verificar la columna 'mes'):
   Departamento de muestra: BUENOS AIRES - 25 DE MAYO
provincia_nombre departamento_nombre           departamento_id cultivo  año  rendimiento  produccion  sup_sembrada  mes
    BUENOS AIRES          25 DE MAYO BUENOS AIRES - 25 DE MAYO     Ajo 1970         3333          10             3    1
    BUENOS AIRES          25 DE MAYO BUENOS AIRES - 25 DE MAYO     Ajo 1970         3333          10             3    2
    BUENOS AIRES          25 DE MAYO BUENOS AIRES - 25 DE MAYO     Ajo 1970         3333          10             3    3
    BUENOS AIRES          25 DE MAYO BUENOS AIRES - 25 DE MAYO     Ajo 1970         3333          10             3    4
    BUENOS AIRES          25 DE MAYO BUENOS AIRES - 25 DE MAYO     Ajo 1970         3333          10             3    5
    BUENOS AIRES          25

In [73]:
print("="*80)
print("PASO 3: CARGA Y PREPARACIÓN DE DATOS DE SUELOS")
print("="*80)

# Cargar el CSV de suelos
df_suelos = pd.read_csv('data/raw/suelos/suelos_argentina_completo.csv')

print(f"\n📊 Información del dataset de suelos:")
print(f"   Registros totales: {len(df_suelos):,}")
print(f"   Columnas: {len(df_suelos.columns)}")

# Verificar provincias
print(f"\n🗺️  Provincias en dataset de suelos: {df_suelos['provincia'].nunique()}")

print("\n⚠️  IMPORTANTE: Los datos de suelos están a nivel de polígono, no de departamento")
print("   Estrategia: Agrupar variables clave de suelos POR PROVINCIA")

# Seleccionar variables numéricas clave de suelos
vars_suelo_numericas = ['ind_prod', 'porc_sue1', 'profund_s1']

# Variables categóricas
vars_suelo_categoricas = ['alcalin_s1']

# Convertir numéricas
vars_numericas_disponibles = []
for v in vars_suelo_numericas:
    if v in df_suelos.columns:
        df_suelos[v] = pd.to_numeric(df_suelos[v], errors='coerce')
        vars_numericas_disponibles.append(v)

print(f"\n   Variables numéricas de suelo: {vars_numericas_disponibles}")

# Para categóricas: usar la moda (valor más frecuente) por provincia
vars_categoricas_disponibles = []
for v in vars_suelo_categoricas:
    if v in df_suelos.columns:
        vars_categoricas_disponibles.append(v)

print(f"   Variables categóricas de suelo: {vars_categoricas_disponibles}")

# Agregar por provincia
# Numéricas: promedio
df_suelos_agg_num = df_suelos.groupby('provincia')[vars_numericas_disponibles].mean().reset_index()
df_suelos_agg_num.columns = ['provincia_nombre'] + [f'suelo_{v}' for v in vars_numericas_disponibles]

# Categóricas: moda (valor más frecuente)
df_suelos_agg_cat = df_suelos.groupby('provincia')[vars_categoricas_disponibles].agg(
    lambda x: x.mode()[0] if len(x.mode()) > 0 else 'Desconocido'
).reset_index()
df_suelos_agg_cat.columns = ['provincia_nombre'] + [f'suelo_{v}' for v in vars_categoricas_disponibles]

# Combinar ambas
df_suelos_agg = pd.merge(df_suelos_agg_num, df_suelos_agg_cat, on='provincia_nombre')

# NORMALIZAR NOMBRES DE PROVINCIAS
# 1. Convertir a title case
df_suelos_agg['provincia_nombre'] = df_suelos_agg['provincia_nombre'].str.title()

# 2. Mapeo manual para casos especiales
mapeo_provincias_suelos = {
    'Santiago Del Estero': 'Santiago del Estero',
    'Tierra Del Fuego E Islas Malvinas': 'Tierra del Fuego'
}
df_suelos_agg['provincia_nombre'] = df_suelos_agg['provincia_nombre'].replace(mapeo_provincias_suelos)

print(f"\n✅ Dataset de suelos agregado por provincia creado")
print(f"   Dimensiones: {df_suelos_agg.shape}")

print(f"\n📋 Provincias normalizadas:")
print(f"   {sorted(df_suelos_agg['provincia_nombre'].unique())}")

print(f"\n📋 Muestra:")
print(df_suelos_agg.head(10))

PASO 3: CARGA Y PREPARACIÓN DE DATOS DE SUELOS

📊 Información del dataset de suelos:
   Registros totales: 7,783
   Columnas: 34

🗺️  Provincias en dataset de suelos: 23

⚠️  IMPORTANTE: Los datos de suelos están a nivel de polígono, no de departamento
   Estrategia: Agrupar variables clave de suelos POR PROVINCIA

   Variables numéricas de suelo: ['ind_prod', 'porc_sue1', 'profund_s1']
   Variables categóricas de suelo: ['alcalin_s1']

✅ Dataset de suelos agregado por provincia creado
   Dimensiones: (23, 5)

📋 Provincias normalizadas:
   ['Buenos Aires', 'Catamarca', 'Chaco', 'Chubut', 'Cordoba', 'Corrientes', 'Entre Rios', 'Formosa', 'Jujuy', 'La Pampa', 'La Rioja', 'Mendoza', 'Misiones', 'Neuquen', 'Rio Negro', 'Salta', 'San Juan', 'San Luis', 'Santa Cruz', 'Santa Fe', 'Santiago del Estero', 'Tierra del Fuego', 'Tucuman']

📋 Muestra:
  provincia_nombre  suelo_ind_prod  suelo_porc_sue1  suelo_profund_s1  \
0     Buenos Aires       34.550898        66.545659         77.879491   
1   

In [74]:
print("="*80)
print("PASO 4: CARGA DEL DATASET DE CLIMA MENSUAL")
print("="*80)

# Cargar el dataset de clima que ya generaste anteriormente
df_clima = pd.read_csv('data/raw/processed/clima_mensual_por_departamento.csv')

print(f"\n📊 Información del dataset de clima:")
print(f"   Registros totales: {len(df_clima):,}")
print(f"   Rango temporal: {df_clima['año'].min()} - {df_clima['año'].max()}")
print(f"   Departamentos únicos: {df_clima['departamento_id'].nunique()}")

print(f"\n📋 Columnas disponibles:")
print(df_clima.columns.tolist())

print(f"\n📋 Muestra de datos de clima:")
print(df_clima.head(10))

print("\n✅ Dataset de clima cargado correctamente")

PASO 4: CARGA DEL DATASET DE CLIMA MENSUAL

📊 Información del dataset de clima:
   Registros totales: 360,778
   Rango temporal: 1969 - 2025
   Departamentos únicos: 529

📋 Columnas disponibles:
['provincia_nombre', 'departamento_nombre', 'departamento_id', 'año', 'mes', 'fecha', 'temperatura_celsius', 'precipitacion_mm']

📋 Muestra de datos de clima:
  provincia_nombre     departamento_nombre  \
0     Buenos Aires  Adolfo Gonzales Chaves   
1     Buenos Aires  Adolfo Gonzales Chaves   
2     Buenos Aires  Adolfo Gonzales Chaves   
3     Buenos Aires  Adolfo Gonzales Chaves   
4     Buenos Aires  Adolfo Gonzales Chaves   
5     Buenos Aires  Adolfo Gonzales Chaves   
6     Buenos Aires  Adolfo Gonzales Chaves   
7     Buenos Aires  Adolfo Gonzales Chaves   
8     Buenos Aires  Adolfo Gonzales Chaves   
9     Buenos Aires  Adolfo Gonzales Chaves   

                         departamento_id   año  mes       fecha  \
0  BUENOS AIRES - ADOLFO GONZALES CHAVES  1969    1  1969-01-01   
1  BU

In [75]:
print("="*80)
print("PASO 4.5: NORMALIZACIÓN DE NOMBRES DE PROVINCIAS")
print("="*80)

# Problema: Los nombres de provincias en el shapefile tienen encoding corrupto
# Solución: Mapear los nombres mal codificados a los correctos

# Crear un diccionario de mapeo de nombres corruptos a nombres correctos
mapeo_provincias = {
    'CÃ³rdoba': 'Cordoba',
    'Entre RÃ\xados': 'Entre Rios',
    'RÃ\xado Negro': 'Rio Negro',
    'TucumÃ¡n': 'Tucuman',
    'Ciudad AutÃ³noma de Buenos Aires': 'Ciudad Autonoma de Buenos Aires'
}

# Aplicar el mapeo al dataset de clima
df_clima['provincia_nombre'] = df_clima['provincia_nombre'].replace(mapeo_provincias)

# También necesitamos corregir el departamento_id
# Función para normalizar el departamento_id
def normalizar_departamento_id(dept_id):
    for corrupto, correcto in mapeo_provincias.items():
        if dept_id.startswith(corrupto):
            return dept_id.replace(corrupto, correcto, 1)
    return dept_id

df_clima['departamento_id'] = df_clima['departamento_id'].apply(normalizar_departamento_id)

print("\n📊 Verificación de provincias únicas en clima:")
print(f"   Total de provincias: {df_clima['provincia_nombre'].nunique()}")
print(f"   Provincias: {sorted(df_clima['provincia_nombre'].unique())}")

print("\n✅ Normalización completada")

PASO 4.5: NORMALIZACIÓN DE NOMBRES DE PROVINCIAS

📊 Verificación de provincias únicas en clima:
   Total de provincias: 24
   Provincias: ['Buenos Aires', 'Catamarca', 'Chaco', 'Chubut', 'Ciudad Autonoma de Buenos Aires', 'Cordoba', 'Corrientes', 'Entre Rios', 'Formosa', 'Jujuy', 'La Pampa', 'La Rioja', 'Mendoza', 'Misiones', 'NeuquÃ©n', 'Rio Negro', 'Salta', 'San Juan', 'San Luis', 'Santa Cruz', 'Santa Fe', 'Santiago del Estero', 'Tierra del Fuego, AntÃ¡rtida e Islas del AtlÃ¡ntico Sur', 'Tucuman']

✅ Normalización completada


In [76]:
print("="*80)
print("PASO 4.6: DIAGNÓSTICO DE DEPARTAMENTOS SIN CLIMA")
print("="*80)

# Verificar qué departamentos de cultivos NO están en clima
deptos_cultivos = set(df_cultivos_mensual['departamento_id'].unique())
deptos_clima = set(df_clima['departamento_id'].unique())

deptos_solo_cultivos = deptos_cultivos - deptos_clima
deptos_solo_clima = deptos_clima - deptos_cultivos
deptos_en_ambos = deptos_cultivos & deptos_clima

print(f"\n📊 Análisis de cobertura:")
print(f"   Departamentos en cultivos: {len(deptos_cultivos)}")
print(f"   Departamentos en clima: {len(deptos_clima)}")
print(f"   Departamentos en AMBOS: {len(deptos_en_ambos)}")
print(f"   Departamentos SOLO en cultivos (sin clima): {len(deptos_solo_cultivos)}")
print(f"   Departamentos SOLO en clima (sin cultivos): {len(deptos_solo_clima)}")

if len(deptos_solo_cultivos) > 0:
    print(f"\n⚠️  Primeros 20 departamentos de cultivos SIN datos de clima:")
    for dept in sorted(deptos_solo_cultivos)[:20]:
        print(f"   - {dept}")

print(f"\n📊 Estadísticas de registros:")
registros_cultivos = len(df_cultivos_mensual)
registros_con_match = df_cultivos_mensual['departamento_id'].isin(deptos_clima).sum()
registros_sin_match = registros_cultivos - registros_con_match

print(f"   Total registros en cultivos: {registros_cultivos:,}")
print(f"   Registros que harán match con clima: {registros_con_match:,} ({registros_con_match/registros_cultivos*100:.1f}%)")
print(f"   Registros que NO harán match: {registros_sin_match:,} ({registros_sin_match/registros_cultivos*100:.1f}%)")

print("\n✅ Diagnóstico completado")
print("   NOTA: Los registros sin match tendrán valores nulos en clima")

PASO 4.6: DIAGNÓSTICO DE DEPARTAMENTOS SIN CLIMA

📊 Análisis de cobertura:
   Departamentos en cultivos: 509
   Departamentos en clima: 529
   Departamentos en AMBOS: 282
   Departamentos SOLO en cultivos (sin clima): 227
   Departamentos SOLO en clima (sin cultivos): 247

⚠️  Primeros 20 departamentos de cultivos SIN datos de clima:
   - BUENOS AIRES - BAHIA BLANCA
   - BUENOS AIRES - BENITO JUAREZ
   - BUENOS AIRES - BOLIVAR
   - BUENOS AIRES - CANUELAS
   - BUENOS AIRES - CAPITAN SARMIENTO
   - BUENOS AIRES - CHASCOMUS
   - BUENOS AIRES - COLON
   - BUENOS AIRES - CORONEL DE MARINA L ROSALES
   - BUENOS AIRES - CORONEL SUAREZ
   - BUENOS AIRES - ESTEBAN ECHEVERRIA
   - BUENOS AIRES - EXALTACION DE LA CRUZ
   - BUENOS AIRES - GENERAL PUEYRREDON
   - BUENOS AIRES - GENERAL RODRIGUEZ
   - BUENOS AIRES - GENERAL SAN MARTIN
   - BUENOS AIRES - GUAMINI
   - BUENOS AIRES - HIPOLITO YRIGOYEN
   - BUENOS AIRES - JUNIN
   - BUENOS AIRES - LANUS
   - BUENOS AIRES - LOBERIA
   - BUENOS AIRES - 

In [77]:
print("="*80)
print("DIAGNÓSTICO: COMPARACIÓN DE FORMATOS")
print("="*80)

# Ver formato de IDs en cultivos
print("\n📋 CULTIVOS - Primeros 5 departamento_id:")
print(df_cultivos_mensual['departamento_id'].head(5).tolist())

# Ver formato de IDs en clima
print("\n🌡️ CLIMA - Primeros 5 departamento_id:")
print(df_clima['departamento_id'].head(5).tolist())

# Buscar coincidencias exactas
cultivos_ids = set(df_cultivos_mensual['departamento_id'].unique())
clima_ids = set(df_clima['departamento_id'].unique())

print(f"\n🔍 Coincidencias:")
print(f"   IDs en cultivos: {len(cultivos_ids)}")
print(f"   IDs en clima: {len(clima_ids)}")
print(f"   IDs que coinciden: {len(cultivos_ids & clima_ids)}")

# Buscar ejemplos de IDs similares
print(f"\n📌 Ejemplos de IDs en CULTIVOS (5 primeros):")
for id in sorted(cultivos_ids)[:5]:
    print(f"   '{id}'")

print(f"\n📌 Ejemplos de IDs en CLIMA (5 primeros):")
for id in sorted(clima_ids)[:5]:
    print(f"   '{id}'")

# Comparar uno específico
print(f"\n🔬 COMPARACIÓN DETALLADA:")
cultivo_ejemplo = sorted(cultivos_ids)[0]
clima_ejemplo = sorted(clima_ids)[0]
print(f"   Cultivo: '{cultivo_ejemplo}'")
print(f"   Clima:   '{clima_ejemplo}'")
print(f"   ¿Son iguales? {cultivo_ejemplo == clima_ejemplo}")

DIAGNÓSTICO: COMPARACIÓN DE FORMATOS

📋 CULTIVOS - Primeros 5 departamento_id:
['BUENOS AIRES - 25 DE MAYO', 'BUENOS AIRES - 25 DE MAYO', 'BUENOS AIRES - 25 DE MAYO', 'BUENOS AIRES - 25 DE MAYO', 'BUENOS AIRES - 25 DE MAYO']

🌡️ CLIMA - Primeros 5 departamento_id:
['BUENOS AIRES - ADOLFO GONZALES CHAVES', 'BUENOS AIRES - ADOLFO GONZALES CHAVES', 'BUENOS AIRES - ADOLFO GONZALES CHAVES', 'BUENOS AIRES - ADOLFO GONZALES CHAVES', 'BUENOS AIRES - ADOLFO GONZALES CHAVES']

🔍 Coincidencias:
   IDs en cultivos: 509
   IDs en clima: 529
   IDs que coinciden: 282

📌 Ejemplos de IDs en CULTIVOS (5 primeros):
   'BUENOS AIRES - 25 DE MAYO'
   'BUENOS AIRES - 9 DE JULIO'
   'BUENOS AIRES - ADOLFO ALSINA'
   'BUENOS AIRES - ADOLFO GONZALES CHAVES'
   'BUENOS AIRES - ALBERTI'

📌 Ejemplos de IDs en CLIMA (5 primeros):
   'BUENOS AIRES - 25 DE MAYO'
   'BUENOS AIRES - 9 DE JULIO'
   'BUENOS AIRES - ADOLFO ALSINA'
   'BUENOS AIRES - ADOLFO GONZALES CHAVES'
   'BUENOS AIRES - ALBERTI'

🔬 COMPARACIÓN DETA

In [78]:
print("="*80)
print("NORMALIZACIÓN DE PROVINCIAS PARA SUELOS")
print("="*80)

# Normalizar provincias en ambos datasets
df_cultivos_mensual['provincia_nombre'] = df_cultivos_mensual['provincia_nombre'].str.title()
df_suelos_agg['provincia_nombre'] = df_suelos_agg['provincia_nombre'].str.title()

# Mapeo manual para casos especiales
mapeo_provincias = {
    'Buenos Aires': 'Buenos Aires',
    'Tierra Del Fuego, Antártida E Islas Del Atlántico Sur': 'Tierra del Fuego'
}

df_cultivos_mensual['provincia_nombre'] = df_cultivos_mensual['provincia_nombre'].replace(mapeo_provincias)
df_suelos_agg['provincia_nombre'] = df_suelos_agg['provincia_nombre'].replace(mapeo_provincias)

# Verificar coincidencias
cultivos_prov = set(df_cultivos_mensual['provincia_nombre'].unique())
suelos_prov = set(df_suelos_agg['provincia_nombre'].unique())
coincidencias = len(cultivos_prov & suelos_prov)

print(f"\n✅ Provincias normalizadas")
print(f"   Provincias en cultivos: {len(cultivos_prov)}")
print(f"   Provincias en suelos: {len(suelos_prov)}")
print(f"   Coincidencias: {coincidencias}")

if coincidencias < 20:
    print(f"\n⚠️  Provincias que NO coinciden:")
    print(f"   Solo en cultivos: {cultivos_prov - suelos_prov}")
    print(f"   Solo en suelos: {suelos_prov - cultivos_prov}")

NORMALIZACIÓN DE PROVINCIAS PARA SUELOS

✅ Provincias normalizadas
   Provincias en cultivos: 23
   Provincias en suelos: 23
   Coincidencias: 23


In [79]:
print("="*80)
print("PASO 5: MERGE FINAL - CULTIVOS + CLIMA + SUELOS")
print("="*80)

print("\n[1/4] Verificando datasets antes del merge...")
print(f"   Cultivos mensual: {len(df_cultivos_mensual):,} registros")
print(f"   Clima mensual: {len(df_clima):,} registros")
print(f"   Suelos (por provincia): {len(df_suelos_agg):,} registros")

print("\n[2/4] Realizando merge de Cultivos + Clima...")
# Merge por departamento_id, año y mes
df_merged = pd.merge(
    df_cultivos_mensual,
    df_clima,
    on=['departamento_id', 'año', 'mes'],
    how='left',
    suffixes=('', '_clima')
)

print(f"   ✓ Registros después de merge cultivos+clima: {len(df_merged):,}")

# Verificar si hay registros sin datos de clima
sin_clima = df_merged['temperatura_celsius'].isna().sum()
print(f"   ⚠️  Registros sin datos de clima: {sin_clima:,} ({sin_clima/len(df_merged)*100:.1f}%)")

print("\n[3/4] Realizando merge con Suelos...")
# Merge por provincia_nombre
df_final = pd.merge(
    df_merged,
    df_suelos_agg,
    on='provincia_nombre',
    how='left'
)

print(f"   ✓ Registros después de merge con suelos: {len(df_final):,}")

# Verificar si hay registros sin datos de suelo
sin_suelo = df_final['suelo_ind_prod'].isna().sum()
print(f"   ⚠️  Registros sin datos de suelo: {sin_suelo:,} ({sin_suelo/len(df_final)*100:.1f}%)")

print("\n[4/4] Limpieza final y selección de columnas...")

# Eliminar columnas duplicadas del merge (si las hay)
# Mantener solo una versión de provincia_nombre y departamento_nombre
if 'provincia_nombre_clima' in df_final.columns:
    df_final = df_final.drop('provincia_nombre_clima', axis=1)
if 'departamento_nombre_clima' in df_final.columns:
    df_final = df_final.drop('departamento_nombre_clima', axis=1)

# Seleccionar columnas finales en el orden deseado
columnas_finales = [
    # Identificadores geográficos
    'provincia_nombre',
    'departamento_nombre', 
    'departamento_id',
    
    # Identificador de cultivo
    'cultivo',
    
    # Temporales
    'año',
    'mes',
    
    # Variable objetivo
    'rendimiento',
    
    # Variables climáticas
    'temperatura_celsius',
    'precipitacion_mm',
    
    # Variables de suelo
    'suelo_ind_prod',
    'suelo_porc_sue1',
    'suelo_profund_s1',
    'suelo_alcalin_s1',  # Variable categórica recuperada
    
    # Variables adicionales (por si acaso)
    'produccion',
    'sup_sembrada'
]

# Verificar que todas las columnas existan
columnas_existentes = [c for c in columnas_finales if c in df_final.columns]
columnas_faltantes = [c for c in columnas_finales if c not in df_final.columns]

if columnas_faltantes:
    print(f"   ⚠️  Columnas no encontradas: {columnas_faltantes}")

df_dataset_final = df_final[columnas_existentes].copy()

print(f"\n✅ Dataset final generado")
print(f"   Dimensiones: {df_dataset_final.shape}")
print(f"   Columnas: {list(df_dataset_final.columns)}")

print("\n📊 Estadísticas del dataset final:")
print(f"   Registros totales: {len(df_dataset_final):,}")
print(f"   Rango temporal: {df_dataset_final['año'].min()} - {df_dataset_final['año'].max()}")
print(f"   Provincias: {df_dataset_final['provincia_nombre'].nunique()}")
print(f"   Departamentos: {df_dataset_final['departamento_id'].nunique()}")
print(f"   Cultivos: {df_dataset_final['cultivo'].nunique()}")

print("\n📋 Muestra del dataset final:")
print(df_dataset_final.head(15))

print("\n📊 Resumen de valores nulos:")
nulos = df_dataset_final.isnull().sum()[df_dataset_final.isnull().sum() > 0]
if len(nulos) > 0:
    for col, count in nulos.items():
        print(f"   {col}: {count:,} ({count/len(df_dataset_final)*100:.1f}%)")

PASO 5: MERGE FINAL - CULTIVOS + CLIMA + SUELOS

[1/4] Verificando datasets antes del merge...
   Cultivos mensual: 1,926,432 registros
   Clima mensual: 360,778 registros
   Suelos (por provincia): 23 registros

[2/4] Realizando merge de Cultivos + Clima...
   ✓ Registros después de merge cultivos+clima: 1,926,432
   ⚠️  Registros sin datos de clima: 853,584 (44.3%)

[3/4] Realizando merge con Suelos...
   ✓ Registros después de merge con suelos: 1,926,432
   ⚠️  Registros sin datos de suelo: 0 (0.0%)

[4/4] Limpieza final y selección de columnas...

✅ Dataset final generado
   Dimensiones: (1926432, 15)
   Columnas: ['provincia_nombre', 'departamento_nombre', 'departamento_id', 'cultivo', 'año', 'mes', 'rendimiento', 'temperatura_celsius', 'precipitacion_mm', 'suelo_ind_prod', 'suelo_porc_sue1', 'suelo_profund_s1', 'suelo_alcalin_s1', 'produccion', 'sup_sembrada']

📊 Estadísticas del dataset final:
   Registros totales: 1,926,432
   Rango temporal: 1970 - 2025
   Provincias: 23
   De

In [80]:
print("="*80)
print("PASO 6: GUARDADO DEL DATASET FINAL")
print("="*80)

# Crear directorio si no existe
import os
os.makedirs('data/processed', exist_ok=True)

# Guardar el dataset final
output_path = 'data/processed/dataset_final_mensual.csv'
df_dataset_final.to_csv(output_path, index=False, encoding='utf-8')

# Información del archivo guardado
file_size_mb = os.path.getsize(output_path) / (1024 * 1024)

print(f"\n💾 Dataset final guardado exitosamente:")
print(f"   Ruta: {output_path}")
print(f"   Tamaño: {file_size_mb:.2f} MB")
print(f"   Registros: {len(df_dataset_final):,}")
print(f"   Columnas: {len(df_dataset_final.columns)}")

print("\n📊 Información detallada del dataset:")
print(f"   Período: {df_dataset_final['año'].min()} - {df_dataset_final['año'].max()} ({df_dataset_final['año'].max() - df_dataset_final['año'].min() + 1} años)")
print(f"   Provincias: {df_dataset_final['provincia_nombre'].nunique()}")
print(f"   Departamentos: {df_dataset_final['departamento_id'].nunique()}")
print(f"   Cultivos: {df_dataset_final['cultivo'].nunique()}")
print(f"   Granularidad temporal: Mensual (12 meses × año)")

print("\n🌾 Top 10 cultivos más representados:")
for cultivo, count in df_dataset_final['cultivo'].value_counts().head(10).items():
    print(f"   {cultivo:25s}: {count:,} registros ({count/len(df_dataset_final)*100:.1f}%)")

print("\n📈 Estadísticas de la variable objetivo (Rendimiento):")
rend_stats = df_dataset_final['rendimiento'].describe()
print(f"   Registros con rendimiento: {df_dataset_final['rendimiento'].notna().sum():,}")
print(f"   Registros sin rendimiento: {df_dataset_final['rendimiento'].isna().sum():,}")
print(f"   Media: {rend_stats['mean']:,.2f}")
print(f"   Mediana (50%): {rend_stats['50%']:,.2f}")
print(f"   Mínimo: {rend_stats['min']:,.2f}")
print(f"   Máximo: {rend_stats['max']:,.2f}")

print("\n🌡️  Estadísticas de variables climáticas:")
temp_validos = df_dataset_final['temperatura_celsius'].notna().sum()
prec_validos = df_dataset_final['precipitacion_mm'].notna().sum()

print(f"   Temperatura (°C):")
print(f"      Registros con datos: {temp_validos:,} ({temp_validos/len(df_dataset_final)*100:.1f}%)")
if temp_validos > 0:
    print(f"      Media: {df_dataset_final['temperatura_celsius'].mean():.2f}°C")
    print(f"      Rango: {df_dataset_final['temperatura_celsius'].min():.2f}°C - {df_dataset_final['temperatura_celsius'].max():.2f}°C")

print(f"   Precipitación (mm):")
print(f"      Registros con datos: {prec_validos:,} ({prec_validos/len(df_dataset_final)*100:.1f}%)")
if prec_validos > 0:
    print(f"      Media: {df_dataset_final['precipitacion_mm'].mean():.2f} mm")
    print(f"      Rango: {df_dataset_final['precipitacion_mm'].min():.2f} - {df_dataset_final['precipitacion_mm'].max():.2f} mm")

print("\n🌱 Estadísticas de variables de suelo:")
suelo_validos = df_dataset_final['suelo_ind_prod'].notna().sum()
print(f"   Registros con datos de suelo: {suelo_validos:,} ({suelo_validos/len(df_dataset_final)*100:.1f}%)")
if suelo_validos > 0:
    print(f"   Índice de productividad promedio: {df_dataset_final['suelo_ind_prod'].mean():.2f}")
    print(f"   Porcentaje suelo 1 promedio: {df_dataset_final['suelo_porc_sue1'].mean():.2f}%")
    print(f"   Profundidad suelo 1 promedio: {df_dataset_final['suelo_profund_s1'].mean():.2f} cm")

print("\n" + "="*80)
print("✅ DATASET FINAL COMPLETADO Y GUARDADO")
print("="*80)
print("\n⚠️  NOTAS IMPORTANTES:")
print("   • ~51% de registros sin datos climáticos (departamentos no cubiertos por shapefile)")
print("   • Variable suelo_alcalin_s1 eliminada (era categórica, no numérica)")
print("   • Datos de suelo agregados a nivel provincial (no departamental)")
print("   • Rendimiento anual replicado en los 12 meses de cada año")
print("\n📝 PRÓXIMOS PASOS RECOMENDADOS:")
print("   1. Ejecutar notebook de verificación (testeo.ipynb)")
print("   2. Análisis exploratorio de datos (EDA)")
print("   3. Decidir estrategia para registros sin clima (eliminar o imputar)")
print("   4. Análisis de correlaciones entre variables")
print("   5. Feature engineering (si es necesario)")
print("   6. Modelado ML con validación temporal (TimeSeriesSplit)")
print("="*80)

PASO 6: GUARDADO DEL DATASET FINAL

💾 Dataset final guardado exitosamente:
   Ruta: data/processed/dataset_final_mensual.csv
   Tamaño: 272.00 MB
   Registros: 1,926,432
   Columnas: 15

📊 Información detallada del dataset:
   Período: 1970 - 2025 (56 años)
   Provincias: 23
   Departamentos: 509
   Cultivos: 41
   Granularidad temporal: Mensual (12 meses × año)

🌾 Top 10 cultivos más representados:
   Maíz                     : 212,700 registros (11.0%)
   Trigo total              : 159,504 registros (8.3%)
   Sorgo                    : 145,620 registros (7.6%)
   Soja total               : 143,796 registros (7.5%)
   Avena                    : 138,408 registros (7.2%)
   Girasol                  : 127,764 registros (6.6%)
   Cebada forrajera         : 78,864 registros (4.1%)
   Soja 1ra                 : 77,688 registros (4.0%)
   Centeno                  : 72,540 registros (3.8%)
   Papa total               : 67,536 registros (3.5%)

📈 Estadísticas de la variable objetivo (Rendimien